# Demo 1B
The purpose of this demonstration is to illustrate some key functionality in pandas, which is covered in Lecture 1.4. After completing this demonstration, you should feel comfortable:
- describing the key functionality of pandas Dataframes and how they compare to pandas Series
- loading data in pandas from a variety of sources
- inspecting metadata underlying the dataframe and generating descriptive statistics
- manipulating data within and adding new data to a dataframe
- combined pandas dataframes either by "stacking" (vertically) or "merging" (horizontally)
- saving final data to disk

### Pandas Dataframes

Built on **numpy**, **pandas** is the popular Python library for data analysis. There are two specific objects we will work with extensively in pandas:
- Pandas __[DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)__: <i>"Two-dimensional, size-mutable, potentially heterogeneous tabular data"</i> (from documentation)
- Pandas __[Series](https://pandas.pydata.org/docs/reference/api/pandas.Series.html)__: <i>"One-dimensional ndarray with axis labels"</i> (from documentation)

As we learn how to structure and analyze data, we will frequently rely on **DataFrames** to collect and organize our data. While we will not use **Series** as often, many pandas methods return **Series**, so it is useful to understand the differences. In addition, a single column from a **DataFrame** is a **Series**.

Let's briefly introduce a pandas **DataFrame**.

In [ ]:
# Pandas is not loaded by default, so we have to import it. The convention for pandas is to load as "pd"
import pandas as pd 

# We'll load a sample dataset that's available on github using the "read_csv()" function.
iris = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')
iris # take a quick glimpse

The *iris* dataset is a very common toy dataset used in various analytics education tasks. We accessed it using the "read_csv" method, which does exactly what you might guess. We used a web address, but a file path works just as well if you have a file saved on your own computer.

### DataFrame Attributes
When loading new dataframes, I often inspect the following attributes:
- shape: provides the dimensions of the dataframe (always *rows* by *columns*)
- index: provides a list of row names (or row labels)
- columns: provides a list of column names

There are several other attributes designed to extract specific pieces of information:
- loc: the preferred way of accessing data within the dataframe
- values: returns the raw data (without labels) in **numpy array** form; this will be useful later for certain methods that require raw numpy data

Let's illustrate:

In [ ]:
# Let's first look at the dataframe's shape:
iris.shape

In [ ]:
# Columns:
iris.columns

In [ ]:
# how to inspect the index?
iris.index

Pandas DataFrames are **mutable**, meaning we can manipulate and change these attributes. For instance, suppose we wanted to convert all column names to upper case:

In [ ]:
newcols = [name.upper() for name in iris.columns]
print(newcols)
iris.columns = newcols
iris

**TIME OUT** for list comprehension!
The syntax we just used to update columns to be upper case is incredibly useful for writing concise, readable code. List comprehension is essentially a *for loop* written in one line, and placing it within a list saves the results of the loop in a list. You can also add some basic logic control, called filtering in this context, with *if statements*. Here are a few additional examples:

In [ ]:
print([col.replace("_","") for col in iris.columns])
print([col.split("_") for col in iris.columns])
print([col for col in iris.columns if col.startswith("SEPAL")])

Now pause and write two additional list comprehensions--one that replaces the underscore with a space and uses capitalized formatting, and one that retains only those columns that capture some kind of width.

In [ ]:
print([col.replace("_"," ").capitalize() for col in iris.columns])
print([col for col in iris.columns if col.endswith("WIDTH")])

**END TIMEOUT**

*Back to pandas:*

As noted, "loc" is used to access specific data within the dataframe. The syntax for loc is as follows:

*df.loc[STMT1,STMT2]*

where:
- STMT1: the condition that describes which *rows* to select; if all rows are desired, pass a ":"
- STMT2: the condition that describes which *columns* to select; if all columns are desired, leave blank or pass a ":"

For those familiar with data.table in R, the syntax may seem familiar. Remember *ROW(S)* then *COLUMN(S)*, always! Note that the *condition* can be be a list of values (also called a *mask*) or a "selection statement".

Let's select sepal dimensions for the setosa species:

In [ ]:
iris_setosa_sepal = iris.loc[iris['SPECIES']=='setosa',['SEPAL_LENGTH','SEPAL_WIDTH']]
iris_setosa_sepal

Alternatively, maybe you just want the first 10 rows (and all columns):

In [ ]:
iris.loc[:10]

Finally, suppose you just wanted the raw data from the smaller dataframe, perhaps to use to train a machine learning model. Use "values":

In [ ]:
print(iris_setosa_sepal.values)

### DataFrame Methods
DataFrames have MANY build in methods. We'll illustrate a few here:
- value_counts() - provide count by unique value; useful for categorical data
- head(*n*) - print the first *n* rows of dataframe (tail focuses on end of dataframe)
- info() - provides information on variables in dataset
- describe() - quick summary statistics for numeric data

Let's illustrate:

In [ ]:
print(iris['SPECIES'].value_counts())

In [ ]:
print(iris.head())

In [ ]:
print(iris.info())

In [ ]:
print(iris.describe())

Pandas DataFrames also have sets of methods that are data-type dependent. The two most common are **string methods** (accessed with **str**) and **datetime methods** (accessed with **dt**). The pandas string methods give you access to most base Python string methods. For example:

In [ ]:
print(iris['SPECIES'].str.capitalize())

Often newcomers to python struggle to distinguish between attributes and methods in code. Remember that methods are like functions, so a simple rule of thumb is that methods require parentheses (e.g., value_counts()), which may or may not include additional parameters, and attributes do not (e.g., df.shape).

The final thing we'll introduce here is the concept of **method chaining**. Method chaining simply means you apply multiple methods sequentially (from left to right). Here's a simple illustration of how to generate descriptive statistics with describe(), and then how to flip rows and columns with transpose():

In [ ]:
print(iris.describe())
print(iris.describe().transpose())

Method chaining is *very* useful when using groupby(), another very powerful (though sometimes tricky) pandas method. The groupby method essentially creates smaller dataframes based on some grouping identifier. Exactly what's done depends on the method chained to groupby. In some instances a much smaller dataframe is returned, such as if you were generating a summary statistic by group. For instance, here's how we'd generate the mean value for each of the four measures by species:

In [ ]:
print(iris.groupby('SPECIES').mean())

Now do the same, but with the standard deviation.

In [ ]:
print(iris.groupby('SPECIES').describe())

In other cases, we may want to analyze one column, or a subset of columns. Suppose we just want to examine SEPAL_LENGTH, or SEPAL_LENGTH and SEPAL_WIDTH:

In [ ]:
print(iris.groupby('SPECIES')['SEPAL_LENGTH'].mean())
print(iris.groupby('SPECIES')[['SEPAL_LENGTH','SEPAL_WIDTH']].mean())

Note the difference between double and single brackets relates to pandas series (one column) vs. a pandas DataFrame (more than one column). If in doubt, use double brackets (though you'll get a dataframe and not series):

In [ ]:
print(iris.groupby('SPECIES')[['SEPAL_LENGTH']].mean())

## Manipulating data in DataFrames ###
So far, we've viewed and summarized data within the dataframe. In most cases we'll be interested in actually manipulating (changing, adding to, combining) the data. We'll divide these operations into three parts:
- Manipulating individual columns
- Applying operations to multiple columns
- Combining entire dataframes by either stacking or merging

### Manipulating individual columns
By manipulating individual columns, I mean altering the values of the column itself with some function or method. This can be accomplished with pandas methods in many cases. For instance, suppose you want the square of SEPAL_LENGTH:

In [ ]:
print(iris['SEPAL_LENGTH'].pow(2)) # "pow" = "power", and argument is power to raise values to

Pandas has many methods, but in some cases we may need to use another function, either one that we write ourselves or one that exists in another package. Numpy functions work very well with pandas data. Let's repeat the exercise above, but using two alternatives, a numpy function and a custom function:

In [ ]:
import numpy as np
print(np.power(iris['SEPAL_LENGTH'],2))

In [ ]:
def custom_power(x):
    return x**2
print(iris['SEPAL_LENGTH'].apply(custom_power))

One subtle point: Many functions we write will not accept a pandas series because the function is not written to operate on a series. This is why I use "apply" above, which *applies* the function to each element of the series. However, because this is simple math, you can pass the column to the function:

In [ ]:
custom_power(iris['SEPAL_LENGTH'])

Finally, in some occasions we may wish to perform some relatively simple operation on each element of a series, but we don't wish to write a separate function. We can do this with a **lambda function**. Anything you can do with a lambda function you can do with a regularly defined function, but lambda functions can save time and make code more readable:

In [ ]:
print(iris['SEPAL_LENGTH'].apply(lambda x: x**2))

### Applying operations to multiple columns
We structure data so that we can analyze it! This often involves applying mathematical operations (or other) on combinations of columns. To illustrate, suppose we want to calculate "Sepal Area". We could accomplish this like this:

In [ ]:
iris['Sepal_Area'] = iris['SEPAL_LENGTH']*iris['SEPAL_WIDTH']
iris['Sepal_Area'].describe()

In that instance, we multipled two pandas Series together with the standard multiplication operator. Python understands we desire pairwise multiplication in that scenario. If we want some other matrix operation, like a dot product, we'd need to use a numpy function.

Now pause and compute "Petal_Area" like we just did with the Sepal measures.

In [ ]:
# Insert your answer here
iris['Petal_Area'] = iris['PETAL_LENGTH']*iris['PETAL_WIDTH']
iris['Petal_Area'].describe()

In some instances, we may wish to combine more than two variables in a more complex expression. We can use the same approach, but writing out the full series name each time is tedious. Pandas has a method called "eval" which greatly simplifies operations like this. For instance, suppose you wished to calculate the ratio of sepal area to petal area (and you had not previously defined them). This would require an expression involving *four* different columns, as well as parentheses to retain the order of operations. We can use eval to simplify this:

In [ ]:
iris['Meaningless_Ratio'] = iris.eval('(SEPAL_LENGTH*SEPAL_WIDTH)/(PETAL_LENGTH*PETAL_WIDTH)')
iris['Meaningless_Ratio'].describe()

We may encounter more advanced operations in later modules, such as one where we wish to divide a set of columns in a dataframe by another value in that dataframe. For instance, maybe our dataframe has word counts, and each column is a unique word. We may wish to divide each of these counts by the total words in the document. We cannot simply use a "/" here unless we explicitly define each new column. There are methods designed to handle these types of operations.

For instance, suppose we wished to divide the first four columns of our dataset (SEPAL_LENGTH, SEPAL_WIDTH, PETAL_LENGTH, PETAL_WIDTH) by the ratio we define above:

In [ ]:
iris.loc[:,iris.columns[:4]].divide(iris['Meaningless_Ratio'],axis=0) 
# Axis = 0 means we're applying this at the row level

### Combining entire dataframes by either stacking or merging
The final example of dataframe manipulation we'll review relates to combining multiple dataframes. Right now, we only have one dataframe, so we'll need to break it apart to illustrate. We're going to do 2 things:
1. Create subsets of data based on species
2. Assign unique identifies and split the Sepal measurements from the Petal measurements

In [ ]:
dfs = []
for spec in iris['SPECIES'].unique():
    print(spec)
    sub = iris.loc[iris['SPECIES']==spec]
    print(sub.head(5))
    dfs.append(sub)

In [ ]:
type(dfs)
dfs[0]

"dfs" now has a list of three different dataframes.

Now for the unique identifiers, we're going to assign using a "groupby()". We'll then split into two datasets.

In [ ]:
# First create identifiers
iris['identifier'] = 1 # placeholder
iris['identifier'] = iris.groupby('SPECIES')['identifier'].cumsum()
iris['identifier'] = iris['SPECIES']+iris['identifier'].astype(str)
iris

In [ ]:
# Now split into sepal and petal
sepal = iris[['identifier','SEPAL_LENGTH','SEPAL_WIDTH']]
petal = iris[['identifier','PETAL_LENGTH','PETAL_WIDTH']]

#### Stacking Data ####
In many exercises, we'll produce a set of smaller dataframes that each have the same columns, but different data (or rows). Once we finish our analysis, we wish to combine all results into a large dataframe. This is the scenario we have in the "dfs" object above.

To combine, we use the pandas function __[concatenate](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)__. This function has one required argument: a list of DataFrames (or Series, and can technically be some kind of mapping to dataframes). The second key argument is "axis", which defaults to 0.

**TIMEOUT**: What is the pandas *axis*?
- Takes two values: 0 or 1
- axis=0: the "index" level; generally used when we want to operate at the row level. Picture "vertical" operations
- axis=1: the "column" level; generally used when we wannt to operate at the column level. Picture "horizontal" operations.
Different functions default to different axis values. *pd.concat* defaults to axis=0, so "row stacking".
**END TIMEOUT**

Now let's demonstrate:

In [ ]:
print(type(dfs)) # dfs is a list
print(type(dfs[0])) # each element is DataFrame
iris2 = pd.concat(dfs,axis=0)
iris2

#### Merging Data ####
There are a number of ways to "merge" data in pandas:
- pd.concat(): If we change the axis to *axis=1*, then concat will "stack horizontally", or merge columns. Note that this procedure **merges on the index**. If the indices of the two DataFrames are not aligned, **you will get erroneous results** (but it will still "work").
- df1.join(df2): DataFrames have a __[join method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)__. This method behaves very similar to pd.merge, but defaults to a "left join" (so df1 is the master DataFrame). Defaults to index-to-index join, but can use any similarly named column
- pd.merge(df1, df2): Pandas has a __[merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)__ that is extremely flexible. Thus function allows you to:
    - specify custom columns to merge on that vary by DataFrame, or merge on index
    - specify any SQL-type join (inner, outer, left, right)
    - add suffixes to overlapping variables

We're going to focus on pd.merge, though we may occasionally use other methods in some instances. Here's how we'd merge our sepal and petal datasets:

In [ ]:
iris3 = pd.merge(sepal,petal,on='identifier',how='inner')
iris3

In [ ]:
petal

## Saving Data ("to disk" or "locally") ##
The last thing we'll cover in this demonstration is how to save data to disk. For now, we'll focus on exporting to CSV, but pandas is capable of writing to numerous other file formats, like excel (xlsx), hdf, txt, "feathers", etc.

When outputting, you typically just need the filepath. If you'd like to save to the same directory in which you have your jupyter notebook saved, you can just run this (assuming we're outputting the original iris dataset):

In [ ]:
iris.to_csv("./iris.csv")

If you inspect this data, you'll notice it has column with numbers 0-149; this is the index. If we don't need that, we can exclude the index:

In [ ]:
iris.to_csv("./iris.csv",index=False)